# Cross-speaker control: does the method measure *who* it was trained on?

The main signal (Perplexity.ipynb, decision 12) says the adapter's perplexity drops 19.8 percentage points more on my replies than on my friends'. A reviewer's objection: that doesn't show it learned *me*. My replies repeat a lot (9.0% of my test replies are exact copies of one of my train messages; StyleClassifier.ipynb), so any adapter trained on me could look like this from catchphrases alone, and the control isn't symmetric: the system prompt names me, and the adapter saw 31 chats of me and none of them.

Here the comparison is made symmetric, inside one chat:

- Two adapters, same recipe, same sessions, about the same number of examples (~1,700 each), same neutral system prompt ("Responde como esta persona en WhatsApp."): one trained on my replies in that chat, one on the other person's.
- Each adapter scored on both speakers' test replies of that chat. Signal of an adapter = drop on the speaker it was trained on − drop on the other one.
- One chat: the one with the most replies from the other person (chat 36 of the v1 data; the other person gave permission). Its test split is the main study's: 7 sessions, 310 replies of mine and 298 of theirs, so the intervals will be wide (`build_speaker.py`).

Runs on the NAS GPU, both trainings with the same settings (micro-batch 2, `--grad-accum 2`, effective batch 4), so they are comparable with each other.

## Prediction (written before running)

Fede: the adapter retrained on me with this chat only should give a signal similar to the main one, but smaller, because it has less data. The other person's adapter should come out similar to mine.

Claude: both positive; mine smaller than the 19.8 of the main adapter, because one chat is less data and in it the two of us write more alike than I do with everyone else.

How it's read: the method measures identity if both adapters give a positive signal (each drops more on the speaker it was trained on) with intervals that exclude 0, and their sizes overlap. If only mine is positive, the main signal is better explained by repetition or by the asymmetric control than by learning a person. With 7 test sessions the intervals will be wide; an inconclusive result is possible and will be reported as such.

In [1]:
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")              # NAS: TLM_PY=/opt/venv/bin/python
D = "data/dataset/speaker"
RUNS = [(36, "me"), (36, "other")]

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line and "█" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

def best(name):
    v = {int(k): x for k, x in json.load(open(R / f"checkpoints_{name}.json"))["valid"].items()}
    s = min(v, key=v.get)
    return s, v[s]

## 1. Datasets (deterministic: the splits of build_dataset.py)

In [2]:
run("uv run python pipeline/build_speaker.py" if "TLM_PY" not in os.environ else f"{PY} pipeline/build_speaker.py")

$ /opt/venv/bin/python build_speaker.py
chat 36    me: train 1787, valid 313, test 310 (other speaker 298), test sessions 7
chat 36 other: train 1758, valid 308, test 298 (other speaker 310), test sessions 7


## 2. Train each adapter (up to 5 epochs), pick the checkpoint on its own validation set, score both speakers on test

In [3]:
for chat, who in RUNS:
    name, data = f"speaker_c{chat}_{who}", f"{D}/c{chat}_{who}"
    if not (ROOT / "adapters" / name / "adapters.safetensors").exists():
        run(f"{PY} scripts/train.py --name {name} --data {data} --epochs 5 --grad-accum 2")
    run(f"{PY} scripts/eval_checkpoints.py adapters/{name} --data {data}/valid.jsonl --out experiments/results/checkpoints_{name}.json")
    step, loss = best(name)
    print(f"{name}: best step {step}, valid {loss:.3f}")
    run(f"{PY} scripts/eval_signal.py --adapter adapters/{name} --step {step} --data {data} --sessions {data}/sessions.json "
        f"--out experiments/results/signal_{name}.json")

$ /opt/venv/bin/python scripts/train.py --name speaker_c36_me --data data/dataset/speaker/c36_me --epochs 5 --grad-accum 2
1787 train examples, batch 4: 2234 steps (5.00 epochs)
/opt/venv/bin/python -m mlx_lm lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --train --data data/dataset/speaker/c36_me --mask-prompt --adapter-path /work/talk-like-me/.nasrun/snap/cross-speaker/adapters/speaker_c36_me --iters 4468 --batch-size 2 --learning-rate 1e-05 --num-layers 16 --max-seq-length 2048 --save-every 200 --steps-per-eval 200 --val-batches 25 --seed 0 --grad-accumulation-steps 2
Loading pretrained model
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen2.5-0.5B-Instruct-4bit                                                          │
│    type            lora · 16 layers · rank 8                                                                         │
│

## Summary

In [4]:
print(f"{'chat':>4} {'trained on':>10} {'drop, own speaker':>18} {'drop, other speaker':>20} {'signal':>8} {'95% CI (sessions)':>20}")
for chat, who in RUNS:
    s = json.load(open(R / f"signal_speaker_c{chat}_{who}.json"))
    print(f"{chat:>4} {who:>10} {s['drop']['me']:>18.1%} {s['drop']['friends']:>20.1%} {s['signal'] * 100:>8.1f} "
          f"[{s['signal_ci'][0] * 100:.1f}, {s['signal_ci'][1] * 100:.1f}]")
print("drop = perplexity drop against the base model; signal = own - other, percentage points")

chat trained on  drop, own speaker  drop, other speaker   signal    95% CI (sessions)
  36         me              52.5%                30.7%     21.8 [13.5, 33.7]
  36      other              69.5%                31.6%     37.9 [32.2, 40.3]
drop = perplexity drop against the base model; signal = own - other, percentage points


## The signal in nats

The base model starts very differently on the two speakers (perplexity 83 on my replies, 125 on theirs), and the percentage signal is not comparable across different starting points (ModelSize.ipynb). In nats per token: the gain against the base model on the speaker the adapter was trained on, minus the gain on the other speaker. The main adapter's signal in nats is 0.41.

In [ ]:
# The signal in nats: (base loss - adapter loss) on the speaker the adapter was trained on, minus the same on the other.
# Independent of where each base model starts (ModelSize.ipynb); the bases differ a lot here (perplexity 83 against 125).
import json
import sys
sys.path.insert(0, "..")
from talk_like_me.stats import by_session, interval, loss_over, resample

for chat, who in [(36, "me"), (36, "other")]:
    s = json.load(open(f"results/signal_speaker_c{chat}_{who}.json"))["per_example"]
    meta = json.load(open(f"../data/dataset/speaker/c{chat}_{who}/sessions.json"))
    sums = {k: by_session(v, meta["me"] if k.endswith("_me") else meta["friends"]) for k, v in s.items()}
    gain = lambda side, c: loss_over(sums[f"base_{side}"], c) - loss_over(sums[f"adapter_{side}"], c)
    sig = lambda c: gain("me", c) - gain("friends", c)
    keys = sorted(set(meta["me"]) | set(meta["friends"]))
    ci = interval([sig(c) for c in resample(keys)])
    print(f"chat {chat}, trained on {who:>5}: gain own {gain('me', keys):.3f}, gain other {gain('friends', keys):.3f}, "
          f"signal {sig(keys):.3f} nats [{ci[0]:.3f}, {ci[1]:.3f}]")


chat 36, trained on    me: gain own 0.744, gain other 0.367, signal 0.377 nats [0.238, 0.563]
chat 36, trained on other: gain own 1.186, gain other 0.380, signal 0.806 nats [0.729, 0.871]


## Conclusion

**Prediction:** Fede, my adapter on this chat gives a signal similar to the main one but smaller (less data), and the other person's similar to mine; Claude, both positive, mine smaller than the main 19.8.

**Both adapters learn the person they were trained on. Right on that; wrong on the sizes.**

| Trained on | Perplexity drop, own speaker | Drop, other speaker | Signal, points (CI by session) | Signal, nats (CI) |
|---|---|---|---|---|
| me (chat 36 only) | 52.5% (83.3 to 39.6) | 30.7% (125.4 to 86.9) | 21.8 [13.5, 33.7] | 0.377 [0.238, 0.563] |
| the other person | 69.5% (125.4 to 38.3) | 31.6% (83.3 to 56.9) | 37.9 [32.2, 40.3] | 0.806 [0.729, 0.871] |

- **The method measures identity, not "whoever is in the prompt".** With the same recipe, data size, sessions and a neutral prompt, each adapter drops much more on the speaker it was trained on. The asymmetry of the main control (an adapter that saw only me, under a prompt that names me) doesn't explain the main signal.
- **My signal on one chat is about the main one, not smaller:** 0.377 nats against 0.41 for the main adapter trained on all my chats (21.8 points against 19.8). One chat with ~1,800 replies is already enough to learn the part that is specifically me in that chat.
- **The other person's is about twice as large** (0.806 nats). The base model finds their replies much harder to start with (perplexity 125 against 83 for mine), so there is more that is specific to them to learn. It says that their writing is more distinctive to this model than mine, not that the method favours them.

**Limits.** One chat, one seed per adapter, 7 test sessions: the intervals resample only 7 units and are rough. The best checkpoints come early (steps 400 and 800 of ~2,200), as expected with ~1,800 examples. Both adapters ran on the NAS with the same settings, so they are comparable with each other, not number by number with the Mac's main adapter.